# Agentic AI for Stock Investment Analysis
## Using Local Ollama LLM

This notebook demonstrates how to build an **agentic AI system** for stock investment analysis using a locally-hosted LLM via Ollama.

### What Makes This "Agentic"?

Traditional stock screeners give you static data. An **agentic** approach means the AI:
- **Reasons** about which data to collect and why
- **Plans** a research workflow (fundamentals → technicals → sentiment → recommendation)
- **Acts** by calling tools to fetch real market data
- **Synthesizes** findings into actionable investment insights
- **Adapts** its analysis based on what it discovers

### Prerequisites
- Ollama installed and running (`ollama serve`)
- A model pulled (e.g., `ollama pull llama3.1` or `ollama pull mistral`)
- Internet connection (for fetching live stock data)

## 1. Setup & Dependencies

In [2]:
%pip install --extra-index-url https://pypi.org/simple yfinance textblob chromadb scipy ollama langchain langchain-ollama numpy pandas requests -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import ssl

# Disable SSL verification for corporate proxy environments
# (corporate firewalls often use self-signed certificates)
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['PYTHONHTTPSVERIFY'] = '0'

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

import ollama
import json
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Optional

# Configure yfinance to not verify SSL (for corporate proxy environments)
try:
    import curl_cffi
    # Patch the default session to disable SSL verification
    _original_session_init = curl_cffi.requests.Session.__init__
    def _patched_init(self, *args, **kwargs):
        kwargs.setdefault('verify', False)
        _original_session_init(self, *args, **kwargs)
    curl_cffi.requests.Session.__init__ = _patched_init
except ImportError:
    pass

# Configuration
MODEL = "llama3.1"  # Change to your preferred Ollama model

# Verify Ollama is running
try:
    models = ollama.list()
    print("Ollama is running. Available models:")
    # Handle both old and new ollama package formats
    model_list = models.get('models', []) if isinstance(models, dict) else models.models
    for model in model_list:
        name = model.get('name', str(model)) if isinstance(model, dict) else model.model
        print(f"  - {name}")
except Exception as e:
    print(f"Error connecting to Ollama: {e}")
    print("Make sure Ollama is running: 'ollama serve'")

Ollama is running. Available models:
  - deepseek-r1:14b
  - deepseek-r1:32b
  - deepseek-r1:1.5b
  - deepseek-r1:8b
  - qwen3-coder:30b
  - qwen2.5:7b
  - llama3.1:8b
  - mistral:latest
  - gpt-oss:20b


## 2. Define Stock Analysis Tools

An agent is only as useful as its tools. For stock analysis, we need tools that can:
- Fetch current and historical price data
- Retrieve fundamental financial metrics
- Compute technical indicators
- Get analyst recommendations and news
- Compare stocks within a sector

In [3]:
def get_stock_price(ticker: str) -> str:
    """Get current stock price and daily change for a given ticker symbol."""
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period="5d")
        if hist.empty:
            return f"No data found for ticker '{ticker}'. Check the symbol."
        
        current = hist['Close'].iloc[-1]
        previous = hist['Close'].iloc[-2] if len(hist) > 1 else current
        change = current - previous
        change_pct = (change / previous) * 100
        
        return json.dumps({
            "ticker": ticker.upper(),
            "current_price": round(current, 2),
            "previous_close": round(previous, 2),
            "change": round(change, 2),
            "change_percent": round(change_pct, 2),
            "currency": stock.info.get("currency", "USD")
        })
    except Exception as e:
        return f"Error fetching price for {ticker}: {str(e)}"


def get_fundamentals(ticker: str) -> str:
    """Get fundamental financial metrics: P/E ratio, market cap, revenue, margins, etc."""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        fundamentals = {
            "ticker": ticker.upper(),
            "company_name": info.get("longName", "N/A"),
            "sector": info.get("sector", "N/A"),
            "industry": info.get("industry", "N/A"),
            "market_cap": info.get("marketCap", "N/A"),
            "pe_ratio": info.get("trailingPE", "N/A"),
            "forward_pe": info.get("forwardPE", "N/A"),
            "peg_ratio": info.get("pegRatio", "N/A"),
            "price_to_book": info.get("priceToBook", "N/A"),
            "dividend_yield": info.get("dividendYield", "N/A"),
            "profit_margin": info.get("profitMargins", "N/A"),
            "revenue_growth": info.get("revenueGrowth", "N/A"),
            "earnings_growth": info.get("earningsGrowth", "N/A"),
            "debt_to_equity": info.get("debtToEquity", "N/A"),
            "return_on_equity": info.get("returnOnEquity", "N/A"),
            "free_cash_flow": info.get("freeCashflow", "N/A"),
            "52_week_high": info.get("fiftyTwoWeekHigh", "N/A"),
            "52_week_low": info.get("fiftyTwoWeekLow", "N/A"),
        }
        return json.dumps(fundamentals)
    except Exception as e:
        return f"Error fetching fundamentals for {ticker}: {str(e)}"


def get_technical_indicators(ticker: str, period: str = "6mo") -> str:
    """Calculate technical indicators: moving averages, RSI, MACD, Bollinger Bands."""
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period=period)
        if hist.empty:
            return f"No historical data for {ticker}"
        
        close = hist['Close']
        
        # Moving Averages
        sma_20 = close.rolling(window=20).mean().iloc[-1]
        sma_50 = close.rolling(window=50).mean().iloc[-1]
        sma_200 = close.rolling(window=200).mean().iloc[-1] if len(close) >= 200 else None
        
        # RSI (14-day)
        delta = close.diff()
        gain = delta.where(delta > 0, 0).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs)).iloc[-1]
        
        # MACD
        ema_12 = close.ewm(span=12, adjust=False).mean()
        ema_26 = close.ewm(span=26, adjust=False).mean()
        macd_line = (ema_12 - ema_26).iloc[-1]
        signal_line = (ema_12 - ema_26).ewm(span=9, adjust=False).mean().iloc[-1]
        
        # Bollinger Bands
        bb_middle = sma_20
        bb_std = close.rolling(window=20).std().iloc[-1]
        bb_upper = bb_middle + (2 * bb_std)
        bb_lower = bb_middle - (2 * bb_std)
        
        current_price = close.iloc[-1]
        
        indicators = {
            "ticker": ticker.upper(),
            "current_price": round(current_price, 2),
            "sma_20": round(sma_20, 2),
            "sma_50": round(sma_50, 2),
            "sma_200": round(sma_200, 2) if sma_200 else "Insufficient data",
            "rsi_14": round(rsi, 2),
            "macd": round(macd_line, 4),
            "macd_signal": round(signal_line, 4),
            "macd_histogram": round(macd_line - signal_line, 4),
            "bollinger_upper": round(bb_upper, 2),
            "bollinger_middle": round(bb_middle, 2),
            "bollinger_lower": round(bb_lower, 2),
            "price_vs_sma50": "ABOVE" if current_price > sma_50 else "BELOW",
            "rsi_signal": "OVERBOUGHT" if rsi > 70 else ("OVERSOLD" if rsi < 30 else "NEUTRAL"),
            "macd_signal_crossover": "BULLISH" if macd_line > signal_line else "BEARISH",
        }
        return json.dumps(indicators)
    except Exception as e:
        return f"Error calculating technicals for {ticker}: {str(e)}"


def get_analyst_recommendations(ticker: str) -> str:
    """Get analyst recommendations and price targets."""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        recs = {
            "ticker": ticker.upper(),
            "recommendation": info.get("recommendationKey", "N/A"),
            "number_of_analysts": info.get("numberOfAnalystOpinions", "N/A"),
            "target_mean_price": info.get("targetMeanPrice", "N/A"),
            "target_high_price": info.get("targetHighPrice", "N/A"),
            "target_low_price": info.get("targetLowPrice", "N/A"),
            "current_price": info.get("currentPrice", "N/A"),
        }
        
        # Calculate upside/downside
        if recs["target_mean_price"] != "N/A" and recs["current_price"] != "N/A":
            upside = ((recs["target_mean_price"] - recs["current_price"]) / recs["current_price"]) * 100
            recs["upside_percent"] = round(upside, 2)
        
        return json.dumps(recs)
    except Exception as e:
        return f"Error fetching recommendations for {ticker}: {str(e)}"


def get_historical_performance(ticker: str, period: str = "1y") -> str:
    """Get historical price performance and volatility metrics."""
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period=period)
        if hist.empty:
            return f"No historical data for {ticker}"
        
        close = hist['Close']
        returns = close.pct_change().dropna()
        
        performance = {
            "ticker": ticker.upper(),
            "period": period,
            "start_price": round(close.iloc[0], 2),
            "end_price": round(close.iloc[-1], 2),
            "total_return_pct": round(((close.iloc[-1] / close.iloc[0]) - 1) * 100, 2),
            "annualized_volatility_pct": round(returns.std() * np.sqrt(252) * 100, 2),
            "max_drawdown_pct": round(((close / close.cummax()) - 1).min() * 100, 2),
            "sharpe_ratio_approx": round((returns.mean() / returns.std()) * np.sqrt(252), 2) if returns.std() > 0 else "N/A",
            "best_day_pct": round(returns.max() * 100, 2),
            "worst_day_pct": round(returns.min() * 100, 2),
            "positive_days_pct": round((returns > 0).sum() / len(returns) * 100, 1),
        }
        return json.dumps(performance)
    except Exception as e:
        return f"Error fetching performance for {ticker}: {str(e)}"


def compare_stocks(tickers: str) -> str:
    """Compare multiple stocks on key metrics. Input: comma-separated tickers (e.g., 'AAPL,MSFT,GOOGL')."""
    try:
        ticker_list = [t.strip().upper() for t in tickers.split(',')]
        comparison = []
        
        for ticker in ticker_list:
            stock = yf.Ticker(ticker)
            info = stock.info
            hist = stock.history(period="1y")
            
            if not hist.empty:
                yearly_return = ((hist['Close'].iloc[-1] / hist['Close'].iloc[0]) - 1) * 100
            else:
                yearly_return = "N/A"
            
            comparison.append({
                "ticker": ticker,
                "name": info.get("shortName", "N/A"),
                "market_cap_B": round(info.get("marketCap", 0) / 1e9, 1),
                "pe_ratio": info.get("trailingPE", "N/A"),
                "forward_pe": info.get("forwardPE", "N/A"),
                "dividend_yield_pct": round(info.get("dividendYield", 0) * 100, 2) if info.get("dividendYield") else 0,
                "1y_return_pct": round(yearly_return, 2) if isinstance(yearly_return, float) else yearly_return,
                "recommendation": info.get("recommendationKey", "N/A"),
            })
        
        return json.dumps(comparison, indent=2)
    except Exception as e:
        return f"Error comparing stocks: {str(e)}"


# Registry of all available tools
TOOLS = {
    "get_stock_price": get_stock_price,
    "get_fundamentals": get_fundamentals,
    "get_technical_indicators": get_technical_indicators,
    "get_analyst_recommendations": get_analyst_recommendations,
    "get_historical_performance": get_historical_performance,
    "compare_stocks": compare_stocks,
}

print("Stock Analysis Tools Defined:")
for name, func in TOOLS.items():
    print(f"  • {name}: {func.__doc__.split(chr(10))[0]}")

Stock Analysis Tools Defined:
  • get_stock_price: Get current stock price and daily change for a given ticker symbol.
  • get_fundamentals: Get fundamental financial metrics: P/E ratio, market cap, revenue, margins, etc.
  • get_technical_indicators: Calculate technical indicators: moving averages, RSI, MACD, Bollinger Bands.
  • get_analyst_recommendations: Get analyst recommendations and price targets.
  • get_historical_performance: Get historical price performance and volatility metrics.
  • compare_stocks: Compare multiple stocks on key metrics. Input: comma-separated tickers (e.g., 'AAPL,MSFT,GOOGL').


## 3. Test the Tools Independently

Before connecting tools to the agent, let's verify they work correctly.

In [ ]:
# Test each tool
print("=== Stock Price ===")
print(get_stock_price("AAPL"))

print("\n=== Fundamentals ===")
fundamentals = json.loads(get_fundamentals("AAPL"))
for key, val in fundamentals.items():
    print(f"  {key}: {val}")

print("\n=== Technical Indicators ===")
technicals = json.loads(get_technical_indicators("AAPL"))
for key, val in technicals.items():
    print(f"  {key}: {val}")

## 4. Build the Stock Research Agent

Now we combine the LLM with our tools into an **investment research agent**. The agent will:
1. Receive a research question about a stock
2. Decide which data to gather
3. Call tools to fetch real market data
4. Reason about the data
5. Provide an investment analysis with recommendation

This follows the **ReAct** (Reasoning + Acting) pattern.

In [ ]:
STOCK_AGENT_SYSTEM_PROMPT = """You are an expert stock investment analyst AI agent. You analyze stocks using fundamental analysis, technical analysis, and market sentiment to provide investment insights.

You have access to these tools:
- get_stock_price(ticker): Get current price and daily change
- get_fundamentals(ticker): Get P/E, market cap, margins, growth metrics
- get_technical_indicators(ticker): Get moving averages, RSI, MACD, Bollinger Bands
- get_analyst_recommendations(ticker): Get analyst consensus and price targets
- get_historical_performance(ticker): Get returns, volatility, Sharpe ratio, drawdown
- compare_stocks(tickers): Compare multiple stocks (comma-separated tickers)

When analyzing a stock, follow this research workflow:
1. Get the current price to understand where it trades now
2. Check fundamentals to assess valuation and financial health
3. Review technicals to understand momentum and timing
4. Check analyst recommendations for consensus view
5. Review historical performance for risk assessment

IMPORTANT RULES:
- Always use tools to get REAL data. Never make up numbers.
- Consider multiple factors before making a recommendation.
- Acknowledge uncertainties and risks.
- This is educational analysis, not financial advice.

Respond in this format:

THOUGHT: [your reasoning about what data you need next]
ACTION: [tool_name]
ACTION_INPUT: [input to the tool]

When you have gathered sufficient data, provide your analysis:

THOUGHT: [synthesize all gathered data]
FINAL_ANSWER: [Your comprehensive investment analysis including:
- Company Overview
- Valuation Assessment
- Technical Outlook
- Risk Factors
- Investment Thesis (Bull/Bear case)
- Overall Rating: Strong Buy / Buy / Hold / Sell / Strong Sell]
"""


def parse_agent_response(text: str):
    """Parse the agent's response to extract thought, action, and final answer."""
    lines = text.strip().split("\n")
    thought = ""
    action = ""
    action_input = ""
    final_answer = ""
    in_final_answer = False

    for line in lines:
        if line.startswith("THOUGHT:"):
            thought = line[len("THOUGHT:"):].strip()
            in_final_answer = False
        elif line.startswith("ACTION:"):
            action = line[len("ACTION:"):].strip()
            in_final_answer = False
        elif line.startswith("ACTION_INPUT:"):
            action_input = line[len("ACTION_INPUT:"):].strip()
            in_final_answer = False
        elif line.startswith("FINAL_ANSWER:"):
            final_answer = line[len("FINAL_ANSWER:"):].strip()
            in_final_answer = True
        elif in_final_answer:
            final_answer += "\n" + line

    return thought, action, action_input, final_answer


def run_stock_agent(query: str, max_iterations: int = 8, verbose: bool = True) -> str:
    """Run the stock research agent on a user query."""
    messages = [
        {"role": "system", "content": STOCK_AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": query}
    ]
    
    gathered_data = []  # Track what data we've collected
    
    if verbose:
        print(f"{'='*70}")
        print(f"STOCK RESEARCH AGENT")
        print(f"{'='*70}")
        print(f"Query: {query}\n")

    for i in range(max_iterations):
        response = ollama.chat(model=MODEL, messages=messages)
        assistant_text = response['message']['content']
        thought, action, action_input, final_answer = parse_agent_response(assistant_text)

        if verbose:
            print(f"--- Step {i+1} ---")
            if thought:
                print(f"  Thought: {thought}")

        if final_answer:
            if verbose:
                print(f"\n{'='*70}")
                print("ANALYSIS COMPLETE")
                print(f"{'='*70}")
                print(final_answer)
            return final_answer

        if action and action in TOOLS:
            if verbose:
                print(f"  Action: {action}({action_input})")
            
            observation = TOOLS[action](action_input)
            gathered_data.append({"tool": action, "input": action_input, "result": observation})
            
            if verbose:
                # Print a condensed version of the observation
                obs_preview = observation[:200] + "..." if len(observation) > 200 else observation
                print(f"  Result: {obs_preview}\n")
            
            messages.append({"role": "assistant", "content": assistant_text})
            messages.append({"role": "user", "content": f"OBSERVATION: {observation}\n\nContinue your analysis. Gather more data if needed or provide your FINAL_ANSWER."})
        elif action:
            if verbose:
                print(f"  Unknown tool: {action}")
            messages.append({"role": "assistant", "content": assistant_text})
            messages.append({"role": "user", "content": f"Error: Unknown tool '{action}'. Available tools: {list(TOOLS.keys())}"})
        else:
            # Model responded without proper format
            if verbose:
                print(f"  Response: {assistant_text[:300]}")
            return assistant_text

    return "Agent reached maximum iterations. Partial data gathered: " + json.dumps(gathered_data, indent=2)

## 5. Run the Agent: Single Stock Analysis

Let's ask the agent to perform a complete analysis of a stock. Watch how it decides which tools to call and in what order.

In [ ]:
# Analyze a single stock - the agent decides what data to gather
result = run_stock_agent(
    "Analyze NVDA (NVIDIA) as a potential investment. "
    "Consider its valuation, technical momentum, and growth prospects. "
    "Is it a good buy at current levels?"
)

## 6. Run the Agent: Comparative Analysis

A key advantage of agentic AI is handling complex, multi-faceted queries that require gathering and synthesizing data from multiple sources.

In [ ]:
# Compare stocks in a sector
result = run_stock_agent(
    "Compare AAPL, MSFT, and GOOGL as investment options. "
    "Which offers the best value right now considering valuation, "
    "growth potential, and technical setup? "
    "Rank them from most attractive to least attractive."
)

## 7. Advanced: Multi-Agent Portfolio Advisor

Real investment analysis involves multiple perspectives. Let's create a **multi-agent system** where:
- **Fundamental Analyst** - focuses on valuation and financials
- **Technical Analyst** - focuses on price action and momentum
- **Risk Analyst** - focuses on downside scenarios and volatility
- **Portfolio Manager** - synthesizes all views into a final recommendation

Each agent has its own expertise and perspective, and the Portfolio Manager arbitrates.

In [ ]:
class SpecialistAgent:
    """A specialist analyst agent with a specific focus area."""
    
    def __init__(self, name: str, system_prompt: str, tools_to_use: list):
        self.name = name
        self.system_prompt = system_prompt
        self.tools_to_use = tools_to_use
    
    def analyze(self, ticker: str, verbose: bool = False) -> str:
        """Run this specialist's analysis on a ticker."""
        # Gather data using assigned tools
        data = {}
        for tool_name in self.tools_to_use:
            if tool_name in TOOLS:
                data[tool_name] = TOOLS[tool_name](ticker)
        
        # Ask the LLM to analyze the data from this specialist's perspective
        analysis_prompt = f"""Analyze {ticker} from your specialist perspective.
        
Here is the data I've gathered:
{json.dumps(data, indent=2)}

Provide a concise analysis (3-5 bullet points) and a rating from 1-10 (10 = extremely bullish).
Format: Start with your bullet points, then end with 'RATING: X/10'"""
        
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": analysis_prompt}
        ]
        
        response = ollama.chat(model=MODEL, messages=messages)
        result = response['message']['content']
        
        if verbose:
            print(f"\n{'─'*50}")
            print(f"{self.name}")
            print(f"{'─'*50}")
            print(result)
        
        return result


# Create specialist agents
fundamental_analyst = SpecialistAgent(
    name="Fundamental Analyst",
    system_prompt="""You are a fundamental stock analyst. You focus on:
    - Valuation metrics (P/E, PEG, Price-to-Book)
    - Financial health (debt, margins, cash flow)
    - Growth trajectory (revenue growth, earnings growth)
    - Competitive position and moat
    You are skeptical of expensive stocks and favor quality at reasonable prices.""",
    tools_to_use=["get_fundamentals", "get_stock_price"]
)

technical_analyst = SpecialistAgent(
    name="Technical Analyst",
    system_prompt="""You are a technical analyst. You focus on:
    - Price trends and moving averages
    - Momentum indicators (RSI, MACD)
    - Support/resistance levels (Bollinger Bands)
    - Volume and price action patterns
    You believe price action tells the story and timing matters.""",
    tools_to_use=["get_technical_indicators", "get_stock_price"]
)

risk_analyst = SpecialistAgent(
    name="Risk Analyst",
    system_prompt="""You are a risk analyst. You focus on:
    - Downside risk and maximum drawdown
    - Volatility and risk-adjusted returns (Sharpe ratio)
    - Concentration risk and sector exposure
    - Macro headwinds and tail risks
    You are naturally cautious and highlight what could go wrong.""",
    tools_to_use=["get_historical_performance", "get_fundamentals"]
)

print("Multi-Agent Portfolio Advisor initialized:")
print("  • Fundamental Analyst - valuation & financial health")
print("  • Technical Analyst - price action & momentum")
print("  • Risk Analyst - downside & volatility")
print("  • Portfolio Manager (LLM) - synthesizes all views")

In [ ]:
def run_multi_agent_analysis(ticker: str, verbose: bool = True) -> str:
    """Run a full multi-agent investment analysis on a stock."""
    
    if verbose:
        print(f"{'='*70}")
        print(f"MULTI-AGENT INVESTMENT ANALYSIS: {ticker.upper()}")
        print(f"{'='*70}")
        print(f"\nGathering specialist opinions...\n")
    
    # Each specialist analyzes the stock independently
    fundamental_view = fundamental_analyst.analyze(ticker, verbose=verbose)
    technical_view = technical_analyst.analyze(ticker, verbose=verbose)
    risk_view = risk_analyst.analyze(ticker, verbose=verbose)
    
    # Portfolio Manager synthesizes all views
    pm_prompt = f"""You are a senior Portfolio Manager making a final investment decision on {ticker.upper()}.

You have received analysis from three specialist analysts:

=== FUNDAMENTAL ANALYST ===
{fundamental_view}

=== TECHNICAL ANALYST ===
{technical_view}

=== RISK ANALYST ===
{risk_view}

Based on all three perspectives, provide your FINAL INVESTMENT DECISION:

1. Summary of key findings across all analyses
2. Where the analysts agree and disagree
3. Your weighted assessment (fundamentals: 40%, technicals: 30%, risk: 30%)
4. Final Rating: STRONG BUY / BUY / HOLD / SELL / STRONG SELL
5. Suggested position sizing (% of portfolio) and time horizon
6. Key catalysts to watch

DISCLAIMER: This is an educational AI analysis exercise, not financial advice."""
    
    messages = [
        {"role": "system", "content": "You are a senior portfolio manager who synthesizes multiple analyst views into a coherent investment recommendation. Be balanced, data-driven, and clear."},
        {"role": "user", "content": pm_prompt}
    ]
    
    response = ollama.chat(model=MODEL, messages=messages)
    final_decision = response['message']['content']
    
    if verbose:
        print(f"\n{'='*70}")
        print("PORTFOLIO MANAGER - FINAL DECISION")
        print(f"{'='*70}")
        print(final_decision)
    
    return final_decision


# Run multi-agent analysis
final_analysis = run_multi_agent_analysis("MSFT")

## 8. Interactive Stock Research Chat

Finally, let's create an interactive conversational agent that maintains context across questions, allowing you to have a natural dialogue about stocks and portfolios.

In [ ]:
class StockResearchChat:
    """Interactive stock research chatbot with memory and tool access."""
    
    def __init__(self, model: str = MODEL):
        self.model = model
        self.conversation_history = []
        self.research_notes = {}  # Store research per ticker
        self.system_prompt = STOCK_AGENT_SYSTEM_PROMPT + """

Additional context: You are in an interactive research session. 
The user may ask follow-up questions about stocks previously discussed.
Reference your previous findings when relevant.
Keep responses focused and actionable."""
    
    def chat(self, user_message: str, verbose: bool = True) -> str:
        """Process a user message and return agent response."""
        messages = [
            {"role": "system", "content": self.system_prompt}
        ]
        
        # Add relevant conversation history (last 5 exchanges)
        for exchange in self.conversation_history[-5:]:
            messages.append({"role": "user", "content": exchange["user"]})
            messages.append({"role": "assistant", "content": exchange["assistant"]})
        
        messages.append({"role": "user", "content": user_message})
        
        if verbose:
            print(f"You: {user_message}")
            print(f"{'─'*50}")
        
        # Agent loop
        for i in range(6):
            response = ollama.chat(model=self.model, messages=messages)
            text = response['message']['content']
            thought, action, action_input, final_answer = parse_agent_response(text)
            
            if final_answer:
                if verbose:
                    if thought:
                        print(f"  [Thinking: {thought}]")
                    print(f"\nAgent: {final_answer}\n")
                self.conversation_history.append({"user": user_message, "assistant": final_answer})
                return final_answer
            
            if action and action in TOOLS:
                if verbose:
                    print(f"  [Fetching: {action}({action_input})]")
                observation = TOOLS[action](action_input)
                messages.append({"role": "assistant", "content": text})
                messages.append({"role": "user", "content": f"OBSERVATION: {observation}\n\nNow provide your FINAL_ANSWER based on this data."})
            else:
                # Direct response without proper format
                if verbose:
                    print(f"\nAgent: {text}\n")
                self.conversation_history.append({"user": user_message, "assistant": text})
                return text
        
        return "I need more iterations to answer that. Try a more specific question."


# Create the chat agent
chat_agent = StockResearchChat()
print("Stock Research Chat initialized!")
print("Ask me anything about stocks, valuations, or portfolio construction.")
print("I'll fetch real market data to inform my answers.\n")

In [ ]:
# Example conversation - run each cell to continue the dialogue
chat_agent.chat("What's the current price and valuation of Tesla (TSLA)?")

In [ ]:
# Follow-up question (the agent remembers context)
chat_agent.chat("How does its technical setup look? Is it overbought or oversold?")

In [ ]:
# Ask for a comparison
chat_agent.chat("Compare Tesla with Ford (F) and GM. Which is better value right now?")

## 10. Additional Dependencies

The following sections require additional packages for sentiment analysis, SEC filings, portfolio optimization, vector databases, and more.

In [4]:
# Install additional packages for advanced features
# !pip install textblob requests scipy chromadb sentence-transformers

# Additional imports
import requests
from textblob import TextBlob
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

print("Additional imports loaded successfully.")

Additional imports loaded successfully.


## 11. News Sentiment Analysis

This tool fetches recent news for a stock and analyzes sentiment using two approaches:
1. **TextBlob** - rule-based NLP for quick polarity scoring
2. **LLM-based** - uses Ollama to provide nuanced sentiment interpretation

The agent uses sentiment signals as one input to its investment thesis.

In [7]:
# First, let's check what yfinance news returns in this version
stock = yf.Ticker("AAPL")
news = stock.news
print(f"News type: {type(news)}")
if news:
    print(f"Number of items: {len(news) if hasattr(news, '__len__') else 'N/A'}")
    if isinstance(news, list) and len(news) > 0:
        print(f"First item type: {type(news[0])}")
        print(f"First item: {news[0]}")
    elif isinstance(news, dict):
        print(f"Keys: {news.keys()}")
        # Print first few items
        for key in list(news.keys())[:3]:
            print(f"  {key}: {str(news[key])[:200]}")
else:
    print("No news returned")

CertificateVerifyError: Failed to perform, curl: (60) SSL certificate problem: self signed certificate in certificate chain. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.

## 12. SEC Filings Retrieval (10-K, 10-Q)

Access SEC EDGAR to retrieve company filings. The SEC provides a free API that returns filing metadata and full-text content. We use the EDGAR Full-Text Search System (EFTS) and the company filings API.

**Important**: SEC EDGAR requires a valid User-Agent header identifying yourself.

In [ ]:
# SEC EDGAR configuration
SEC_HEADERS = {
    "User-Agent": "StockAnalysisBot research@example.com",  # SEC requires identification
    "Accept-Encoding": "gzip, deflate",
}

def get_company_cik(ticker: str) -> Optional[str]:
    """Look up a company's CIK number from its ticker symbol."""
    try:
        url = "https://www.sec.gov/files/company_tickers.json"
        response = requests.get(url, headers=SEC_HEADERS, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        ticker_upper = ticker.upper()
        for entry in data.values():
            if entry.get("ticker", "").upper() == ticker_upper:
                # CIK needs to be zero-padded to 10 digits
                return str(entry["cik_str"]).zfill(10)
        return None
    except Exception as e:
        return None


def get_sec_filings(ticker: str, filing_type: str = "10-K") -> str:
    """Retrieve recent SEC filings (10-K annual or 10-Q quarterly) for a company.
    Input: ticker symbol. Optional filing_type: '10-K' (annual) or '10-Q' (quarterly).
    Returns filing dates, links, and key excerpts."""
    try:
        # Parse filing type from input if combined (e.g., "AAPL,10-Q")
        parts = ticker.split(",")
        actual_ticker = parts[0].strip()
        if len(parts) > 1:
            filing_type = parts[1].strip()
        
        # Get CIK
        cik = get_company_cik(actual_ticker)
        if not cik:
            return json.dumps({"error": f"Could not find CIK for ticker '{actual_ticker}'"})
        
        # Query SEC EDGAR for filings
        url = f"https://data.sec.gov/submissions/CIK{cik}.json"
        response = requests.get(url, headers=SEC_HEADERS, timeout=15)
        response.raise_for_status()
        data = response.json()
        
        company_name = data.get("name", "Unknown")
        recent_filings = data.get("filings", {}).get("recent", {})
        
        if not recent_filings:
            return json.dumps({"error": "No recent filings found"})
        
        # Filter by filing type
        forms = recent_filings.get("form", [])
        dates = recent_filings.get("filingDate", [])
        accessions = recent_filings.get("accessionNumber", [])
        primary_docs = recent_filings.get("primaryDocument", [])
        descriptions = recent_filings.get("primaryDocDescription", [])
        
        filtered_filings = []
        for i in range(len(forms)):
            if forms[i] == filing_type:
                accession_clean = accessions[i].replace("-", "")
                filing_url = f"https://www.sec.gov/Archives/edgar/data/{cik.lstrip('0')}/{accession_clean}/{primary_docs[i]}"
                
                filtered_filings.append({
                    "form_type": forms[i],
                    "filing_date": dates[i],
                    "description": descriptions[i] if i < len(descriptions) else "",
                    "url": filing_url,
                })
                
                if len(filtered_filings) >= 5:  # Get last 5 filings
                    break
        
        result = {
            "ticker": actual_ticker.upper(),
            "company_name": company_name,
            "cik": cik,
            "filing_type": filing_type,
            "filings_found": len(filtered_filings),
            "recent_filings": filtered_filings,
        }
        return json.dumps(result, indent=2)
    except Exception as e:
        return f"Error retrieving SEC filings for {ticker}: {str(e)}"


def get_filing_summary(ticker: str) -> str:
    """Get a summary of a company's most recent 10-K filing using the LLM.
    Fetches key financial data from SEC and asks the LLM to summarize."""
    try:
        # Get filings list
        filings_data = json.loads(get_sec_filings(ticker, "10-K"))
        
        if "error" in filings_data:
            return json.dumps(filings_data)
        
        # Also get yfinance fundamentals for context
        stock = yf.Ticker(ticker)
        info = stock.info
        
        # Build context for LLM
        context = f"""Company: {filings_data['company_name']} ({ticker.upper()})
CIK: {filings_data['cik']}
Most Recent 10-K Filing Date: {filings_data['recent_filings'][0]['filing_date'] if filings_data['recent_filings'] else 'N/A'}

Key Financial Data (from latest available):
- Revenue: ${info.get('totalRevenue', 'N/A'):,}
- Net Income: ${info.get('netIncomeToCommon', 'N/A'):,}
- Total Debt: ${info.get('totalDebt', 'N/A'):,}
- Total Cash: ${info.get('totalCash', 'N/A'):,}
- Operating Margins: {info.get('operatingMargins', 'N/A')}
- Revenue Growth: {info.get('revenueGrowth', 'N/A')}
- Employees: {info.get('fullTimeEmployees', 'N/A')}
- Business Summary: {info.get('longBusinessSummary', 'N/A')[:500]}"""

        # Ask LLM to summarize as if reading a 10-K
        prompt = f"""Based on this financial data for {ticker.upper()}, provide a 10-K style summary covering:
1. Business Overview (what does the company do)
2. Financial Highlights (revenue, profitability, growth)
3. Balance Sheet Health (debt vs cash)
4. Key Risks (based on the business and financial position)
5. Growth Outlook

Data:
{context}

Provide a concise, professional analyst summary."""

        response = ollama.chat(model=MODEL, messages=[
            {"role": "system", "content": "You are a financial analyst summarizing SEC filings. Be factual and concise."},
            {"role": "user", "content": prompt}
        ])
        
        result = {
            "ticker": ticker.upper(),
            "company_name": filings_data['company_name'],
            "latest_10k_date": filings_data['recent_filings'][0]['filing_date'] if filings_data['recent_filings'] else "N/A",
            "filing_url": filings_data['recent_filings'][0]['url'] if filings_data['recent_filings'] else "N/A",
            "summary": response['message']['content']
        }
        return json.dumps(result, indent=2)
    except Exception as e:
        return f"Error summarizing filing for {ticker}: {str(e)}"


# Test SEC filings retrieval
print("=== SEC Filings Retrieval ===")
filings = get_sec_filings("AAPL", "10-K")
parsed = json.loads(filings)
print(f"Company: {parsed.get('company_name', 'N/A')}")
print(f"CIK: {parsed.get('cik', 'N/A')}")
print(f"Filing Type: {parsed.get('filing_type', 'N/A')}")
print(f"Filings Found: {parsed.get('filings_found', 0)}")
print("\nRecent filings:")
for f in parsed.get('recent_filings', [])[:3]:
    print(f"  {f['filing_date']} - {f['form_type']} - {f['url'][:80]}...")

## 13. Portfolio Optimizer (Modern Portfolio Theory)

Implements **Markowitz Mean-Variance Optimization** to find optimal portfolio allocations. Given a set of stocks, the optimizer:
- Calculates expected returns and covariance matrix from historical data
- Finds the **efficient frontier** (max return for each risk level)
- Identifies the **maximum Sharpe ratio** portfolio and **minimum variance** portfolio
- Provides allocation weights the agent can recommend

In [ ]:
def optimize_portfolio(tickers: str, period: str = "2y", risk_free_rate: float = 0.05) -> str:
    """Optimize portfolio allocation using Modern Portfolio Theory (Mean-Variance Optimization).
    Input: comma-separated ticker symbols (e.g., 'AAPL,MSFT,GOOGL,AMZN').
    Returns optimal weights for max Sharpe ratio and min variance portfolios."""
    try:
        ticker_list = [t.strip().upper() for t in tickers.split(',')]
        
        if len(ticker_list) < 2:
            return json.dumps({"error": "Need at least 2 tickers for portfolio optimization"})
        
        # Download historical data
        data = yf.download(ticker_list, period=period, progress=False)['Close']
        
        if data.empty:
            return json.dumps({"error": "Could not download price data"})
        
        # Drop any tickers with missing data
        data = data.dropna(axis=1, how='all').dropna()
        valid_tickers = list(data.columns)
        
        if len(valid_tickers) < 2:
            return json.dumps({"error": "Insufficient data for optimization"})
        
        # Calculate daily returns
        returns = data.pct_change().dropna()
        
        # Annualized return and covariance
        mean_returns = returns.mean() * 252
        cov_matrix = returns.cov() * 252
        n_assets = len(valid_tickers)
        
        def portfolio_performance(weights):
            """Calculate portfolio return and volatility."""
            port_return = np.dot(weights, mean_returns)
            port_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
            return port_return, port_volatility
        
        def neg_sharpe_ratio(weights):
            """Negative Sharpe ratio (for minimization)."""
            p_return, p_volatility = portfolio_performance(weights)
            return -(p_return - risk_free_rate) / p_volatility
        
        def portfolio_volatility(weights):
            """Portfolio volatility (for minimum variance)."""
            return np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        
        # Constraints: weights sum to 1
        constraints = {'type': 'eq', 'fun': lambda x: np.sum(x) - 1}
        # Bounds: no short selling (0 to 1 for each weight)
        bounds = tuple((0, 1) for _ in range(n_assets))
        # Initial guess: equal weight
        init_weights = np.array([1/n_assets] * n_assets)
        
        # Optimize for Maximum Sharpe Ratio
        max_sharpe_result = minimize(
            neg_sharpe_ratio, init_weights,
            method='SLSQP', bounds=bounds, constraints=constraints
        )
        max_sharpe_weights = max_sharpe_result.x
        max_sharpe_return, max_sharpe_vol = portfolio_performance(max_sharpe_weights)
        max_sharpe = (max_sharpe_return - risk_free_rate) / max_sharpe_vol
        
        # Optimize for Minimum Variance
        min_var_result = minimize(
            portfolio_volatility, init_weights,
            method='SLSQP', bounds=bounds, constraints=constraints
        )
        min_var_weights = min_var_result.x
        min_var_return, min_var_vol = portfolio_performance(min_var_weights)
        
        # Equal weight portfolio for comparison
        eq_return, eq_vol = portfolio_performance(init_weights)
        eq_sharpe = (eq_return - risk_free_rate) / eq_vol
        
        # Individual stock metrics
        individual_metrics = []
        for i, ticker in enumerate(valid_tickers):
            individual_metrics.append({
                "ticker": ticker,
                "annual_return_pct": round(mean_returns.iloc[i] * 100, 2),
                "annual_volatility_pct": round(np.sqrt(cov_matrix.iloc[i, i]) * 100, 2),
                "sharpe_ratio": round((mean_returns.iloc[i] - risk_free_rate) / np.sqrt(cov_matrix.iloc[i, i]), 2),
            })
        
        result = {
            "tickers": valid_tickers,
            "period": period,
            "risk_free_rate": risk_free_rate,
            "max_sharpe_portfolio": {
                "weights": {valid_tickers[i]: round(max_sharpe_weights[i], 4) for i in range(n_assets)},
                "expected_annual_return_pct": round(max_sharpe_return * 100, 2),
                "annual_volatility_pct": round(max_sharpe_vol * 100, 2),
                "sharpe_ratio": round(max_sharpe, 2),
            },
            "min_variance_portfolio": {
                "weights": {valid_tickers[i]: round(min_var_weights[i], 4) for i in range(n_assets)},
                "expected_annual_return_pct": round(min_var_return * 100, 2),
                "annual_volatility_pct": round(min_var_vol * 100, 2),
                "sharpe_ratio": round((min_var_return - risk_free_rate) / min_var_vol, 2),
            },
            "equal_weight_portfolio": {
                "weights": {t: round(1/n_assets, 4) for t in valid_tickers},
                "expected_annual_return_pct": round(eq_return * 100, 2),
                "annual_volatility_pct": round(eq_vol * 100, 2),
                "sharpe_ratio": round(eq_sharpe, 2),
            },
            "individual_stocks": individual_metrics,
            "correlation_matrix": returns.corr().round(3).to_dict(),
        }
        return json.dumps(result, indent=2)
    except Exception as e:
        return f"Error in portfolio optimization: {str(e)}"


# Test portfolio optimizer
print("=== Portfolio Optimization (Modern Portfolio Theory) ===\n")
opt_result = optimize_portfolio("AAPL,MSFT,GOOGL,AMZN,META")
parsed = json.loads(opt_result)

print("Max Sharpe Ratio Portfolio:")
for ticker, weight in parsed['max_sharpe_portfolio']['weights'].items():
    if weight > 0.01:
        print(f"  {ticker}: {weight*100:.1f}%")
print(f"  Expected Return: {parsed['max_sharpe_portfolio']['expected_annual_return_pct']}%")
print(f"  Volatility: {parsed['max_sharpe_portfolio']['annual_volatility_pct']}%")
print(f"  Sharpe Ratio: {parsed['max_sharpe_portfolio']['sharpe_ratio']}")

print("\nMin Variance Portfolio:")
for ticker, weight in parsed['min_variance_portfolio']['weights'].items():
    if weight > 0.01:
        print(f"  {ticker}: {weight*100:.1f}%")
print(f"  Expected Return: {parsed['min_variance_portfolio']['expected_annual_return_pct']}%")
print(f"  Volatility: {parsed['min_variance_portfolio']['annual_volatility_pct']}%")

## 14. Backtesting Engine

A simple but functional backtesting framework that validates trading strategies against historical data. Supports:
- **Moving Average Crossover** strategies (Golden Cross / Death Cross)
- **RSI-based** mean reversion strategies
- **Buy-and-Hold** baseline comparison
- Performance metrics: total return, Sharpe ratio, max drawdown, win rate

In [ ]:
def backtest_strategy(ticker: str, strategy: str = "sma_crossover", period: str = "2y") -> str:
    """Backtest a trading strategy on historical data.
    Input: ticker, strategy type ('sma_crossover', 'rsi_mean_reversion', 'buy_and_hold').
    Returns performance metrics compared to buy-and-hold baseline."""
    try:
        # Parse input - support "AAPL,rsi_mean_reversion" format
        parts = ticker.split(",")
        actual_ticker = parts[0].strip()
        if len(parts) > 1:
            strategy = parts[1].strip()
        
        stock = yf.Ticker(actual_ticker)
        hist = stock.history(period=period)
        
        if hist.empty or len(hist) < 60:
            return json.dumps({"error": f"Insufficient data for {actual_ticker} (need at least 60 days)"})
        
        close = hist['Close'].copy()
        dates = hist.index
        
        # Initialize signals (1 = long, 0 = out of market)
        signals = pd.Series(0, index=dates)
        
        if strategy == "sma_crossover":
            # Golden Cross / Death Cross: SMA 50 crosses above/below SMA 200
            sma_short = close.rolling(window=20).mean()
            sma_long = close.rolling(window=50).mean()
            
            # Generate signals
            for i in range(1, len(close)):
                if pd.notna(sma_short.iloc[i]) and pd.notna(sma_long.iloc[i]):
                    if sma_short.iloc[i] > sma_long.iloc[i]:
                        signals.iloc[i] = 1  # Buy signal
                    else:
                        signals.iloc[i] = 0  # Sell signal
            
            strategy_name = "SMA 20/50 Crossover"
            
        elif strategy == "rsi_mean_reversion":
            # Buy when RSI < 30 (oversold), Sell when RSI > 70 (overbought)
            delta = close.diff()
            gain = delta.where(delta > 0, 0).rolling(window=14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
            rs = gain / loss
            rsi = 100 - (100 / (1 + rs))
            
            position = 0
            for i in range(14, len(close)):
                if pd.notna(rsi.iloc[i]):
                    if rsi.iloc[i] < 30:
                        position = 1  # Buy (oversold)
                    elif rsi.iloc[i] > 70:
                        position = 0  # Sell (overbought)
                signals.iloc[i] = position
            
            strategy_name = "RSI Mean Reversion (30/70)"
            
        elif strategy == "buy_and_hold":
            signals[:] = 1
            strategy_name = "Buy and Hold"
            
        else:
            return json.dumps({"error": f"Unknown strategy '{strategy}'. Use: sma_crossover, rsi_mean_reversion, buy_and_hold"})
        
        # Calculate returns
        daily_returns = close.pct_change().fillna(0)
        
        # Strategy returns (only when signal = 1)
        strategy_returns = daily_returns * signals.shift(1).fillna(0)
        
        # Buy and hold returns (baseline)
        buyhold_returns = daily_returns
        
        # Cumulative returns
        strategy_cumulative = (1 + strategy_returns).cumprod()
        buyhold_cumulative = (1 + buyhold_returns).cumprod()
        
        # Performance metrics
        total_strategy_return = (strategy_cumulative.iloc[-1] - 1) * 100
        total_buyhold_return = (buyhold_cumulative.iloc[-1] - 1) * 100
        
        # Annualized metrics
        trading_days = len(close)
        years = trading_days / 252
        
        ann_strategy_return = ((1 + total_strategy_return/100) ** (1/years) - 1) * 100
        ann_buyhold_return = ((1 + total_buyhold_return/100) ** (1/years) - 1) * 100
        
        # Volatility
        strategy_vol = strategy_returns.std() * np.sqrt(252) * 100
        buyhold_vol = buyhold_returns.std() * np.sqrt(252) * 100
        
        # Sharpe Ratio (assuming 5% risk-free rate)
        rf_daily = 0.05 / 252
        strategy_sharpe = ((strategy_returns.mean() - rf_daily) / strategy_returns.std()) * np.sqrt(252) if strategy_returns.std() > 0 else 0
        buyhold_sharpe = ((buyhold_returns.mean() - rf_daily) / buyhold_returns.std()) * np.sqrt(252) if buyhold_returns.std() > 0 else 0
        
        # Max Drawdown
        strategy_peak = strategy_cumulative.cummax()
        strategy_drawdown = ((strategy_cumulative - strategy_peak) / strategy_peak).min() * 100
        
        buyhold_peak = buyhold_cumulative.cummax()
        buyhold_drawdown = ((buyhold_cumulative - buyhold_peak) / buyhold_peak).min() * 100
        
        # Trade statistics
        signal_changes = signals.diff().fillna(0)
        n_trades = int((signal_changes != 0).sum() / 2)
        time_in_market = signals.mean() * 100
        
        # Win rate (profitable trades)
        trade_returns = []
        in_trade = False
        trade_start_value = 0
        for i in range(1, len(signals)):
            if signals.iloc[i] == 1 and signals.iloc[i-1] == 0:
                in_trade = True
                trade_start_value = strategy_cumulative.iloc[i]
            elif signals.iloc[i] == 0 and signals.iloc[i-1] == 1 and in_trade:
                in_trade = False
                trade_return = (strategy_cumulative.iloc[i] / trade_start_value - 1) if trade_start_value > 0 else 0
                trade_returns.append(trade_return)
        
        win_rate = (sum(1 for r in trade_returns if r > 0) / len(trade_returns) * 100) if trade_returns else 0
        
        result = {
            "ticker": actual_ticker.upper(),
            "strategy": strategy_name,
            "period": period,
            "trading_days": trading_days,
            "strategy_performance": {
                "total_return_pct": round(total_strategy_return, 2),
                "annualized_return_pct": round(ann_strategy_return, 2),
                "annualized_volatility_pct": round(strategy_vol, 2),
                "sharpe_ratio": round(strategy_sharpe, 2),
                "max_drawdown_pct": round(strategy_drawdown, 2),
            },
            "buy_and_hold_baseline": {
                "total_return_pct": round(total_buyhold_return, 2),
                "annualized_return_pct": round(ann_buyhold_return, 2),
                "annualized_volatility_pct": round(buyhold_vol, 2),
                "sharpe_ratio": round(buyhold_sharpe, 2),
                "max_drawdown_pct": round(buyhold_drawdown, 2),
            },
            "trade_statistics": {
                "number_of_trades": n_trades,
                "time_in_market_pct": round(time_in_market, 1),
                "win_rate_pct": round(win_rate, 1),
                "avg_trade_return_pct": round(np.mean(trade_returns) * 100, 2) if trade_returns else 0,
            },
            "outperformance_pct": round(total_strategy_return - total_buyhold_return, 2),
            "verdict": "STRATEGY OUTPERFORMS" if total_strategy_return > total_buyhold_return else "BUY & HOLD WINS"
        }
        return json.dumps(result, indent=2)
    except Exception as e:
        return f"Error in backtesting {ticker}: {str(e)}"


# Test backtesting
print("=== Backtesting: SMA Crossover on AAPL ===\n")
bt_result = backtest_strategy("AAPL", "sma_crossover", "2y")
parsed = json.loads(bt_result)

print(f"Strategy: {parsed['strategy']}")
print(f"Period: {parsed['period']} ({parsed['trading_days']} trading days)")
print(f"\nStrategy Performance:")
print(f"  Total Return: {parsed['strategy_performance']['total_return_pct']}%")
print(f"  Sharpe Ratio: {parsed['strategy_performance']['sharpe_ratio']}")
print(f"  Max Drawdown: {parsed['strategy_performance']['max_drawdown_pct']}%")
print(f"\nBuy & Hold Baseline:")
print(f"  Total Return: {parsed['buy_and_hold_baseline']['total_return_pct']}%")
print(f"  Sharpe Ratio: {parsed['buy_and_hold_baseline']['sharpe_ratio']}")
print(f"\nVerdict: {parsed['verdict']}")
print(f"Trades: {parsed['trade_statistics']['number_of_trades']} | Win Rate: {parsed['trade_statistics']['win_rate_pct']}%")

print("\n\n=== Backtesting: RSI Mean Reversion on AAPL ===\n")
bt_rsi = backtest_strategy("AAPL", "rsi_mean_reversion", "2y")
parsed_rsi = json.loads(bt_rsi)
print(f"Strategy: {parsed_rsi['strategy']}")
print(f"Total Return: {parsed_rsi['strategy_performance']['total_return_pct']}%")
print(f"Verdict: {parsed_rsi['verdict']}")

## 15. Vector Database for Historical Pattern Matching

Uses **ChromaDB** (local vector database) to store and retrieve similar historical price patterns. This enables the agent to find periods in the past that resemble current market conditions and learn from what happened next.

The workflow:
1. Convert price patterns into embeddings (using Ollama's embedding model or numerical features)
2. Store patterns in ChromaDB with metadata (date, ticker, what happened next)
3. Query for similar patterns to inform predictions

In [ ]:
import chromadb
from chromadb.config import Settings

# Initialize ChromaDB (local, in-memory for this demo)
chroma_client = chromadb.Client()

# Create a collection for stock patterns
try:
    chroma_client.delete_collection("stock_patterns")
except:
    pass
pattern_collection = chroma_client.create_collection(
    name="stock_patterns",
    metadata={"description": "Historical stock price patterns and outcomes"}
)


def extract_pattern_features(prices: pd.Series, window: int = 20) -> list:
    """Convert a price window into a normalized feature vector for similarity comparison."""
    if len(prices) < window:
        return []
    
    # Normalize prices to percentage changes from start
    normalized = ((prices / prices.iloc[0]) - 1).tolist()
    
    # Add technical features
    returns = prices.pct_change().dropna()
    volatility = returns.std()
    trend = (prices.iloc[-1] / prices.iloc[0]) - 1  # Overall trend
    
    # Combine into feature vector (fixed length)
    features = normalized[-window:]  # Last N normalized prices
    features.extend([volatility, trend])
    
    return [float(f) for f in features]


def build_pattern_database(ticker: str, window: int = 20, period: str = "5y") -> str:
    """Build a database of historical price patterns for a stock.
    Stores sliding windows of price patterns with their subsequent outcomes."""
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period=period)
        
        if hist.empty or len(hist) < window + 20:
            return json.dumps({"error": f"Insufficient history for {ticker}"})
        
        close = hist['Close']
        patterns_stored = 0
        
        ids = []
        embeddings = []
        metadatas = []
        documents = []
        
        # Create sliding windows
        step = 5  # Step size to avoid too many overlapping patterns
        for i in range(0, len(close) - window - 10, step):
            pattern_prices = close.iloc[i:i+window]
            future_prices = close.iloc[i+window:i+window+10]
            
            features = extract_pattern_features(pattern_prices, window)
            if not features or len(features) != window + 2:
                continue
            
            # Calculate outcome (what happened in the next 10 days)
            future_return = (future_prices.iloc[-1] / future_prices.iloc[0] - 1) * 100
            
            if future_return > 3:
                outcome = "STRONG_UP"
            elif future_return > 1:
                outcome = "MILD_UP"
            elif future_return < -3:
                outcome = "STRONG_DOWN"
            elif future_return < -1:
                outcome = "MILD_DOWN"
            else:
                outcome = "FLAT"
            
            pattern_date = pattern_prices.index[-1].strftime("%Y-%m-%d")
            
            ids.append(f"{ticker}_{pattern_date}_{i}")
            embeddings.append(features)
            metadatas.append({
                "ticker": ticker.upper(),
                "pattern_end_date": pattern_date,
                "future_return_pct": round(future_return, 2),
                "outcome": outcome,
                "pattern_trend": "UP" if features[-1] > 0.02 else ("DOWN" if features[-1] < -0.02 else "FLAT"),
            })
            documents.append(
                f"{ticker} pattern ending {pattern_date}: "
                f"trend={'up' if features[-1] > 0 else 'down'}, "
                f"vol={features[-2]:.4f}, "
                f"outcome={outcome} ({future_return:.1f}% in next 10 days)"
            )
            patterns_stored += 1
        
        # Store in ChromaDB
        if ids:
            pattern_collection.add(
                ids=ids,
                embeddings=embeddings,
                metadatas=metadatas,
                documents=documents
            )
        
        return json.dumps({
            "ticker": ticker.upper(),
            "patterns_stored": patterns_stored,
            "window_size": window,
            "history_period": period,
            "status": "Database built successfully"
        })
    except Exception as e:
        return f"Error building pattern database: {str(e)}"


def find_similar_patterns(ticker: str, window: int = 20) -> str:
    """Find historical patterns similar to the current price action.
    Returns the top matches and what happened after those similar patterns."""
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period="2mo")  # Get recent data
        
        if hist.empty or len(hist) < window:
            return json.dumps({"error": f"Insufficient recent data for {ticker}"})
        
        # Get current pattern
        recent_prices = hist['Close'].iloc[-window:]
        current_features = extract_pattern_features(recent_prices, window)
        
        if not current_features or len(current_features) != window + 2:
            return json.dumps({"error": "Could not extract features from current pattern"})
        
        # Check if we have patterns in the database
        collection_count = pattern_collection.count()
        if collection_count == 0:
            # Auto-build database if empty
            build_pattern_database(ticker)
            collection_count = pattern_collection.count()
            if collection_count == 0:
                return json.dumps({"error": "No patterns in database. Run build_pattern_database first."})
        
        # Query for similar patterns
        results = pattern_collection.query(
            query_embeddings=[current_features],
            n_results=min(10, collection_count),
        )
        
        if not results['ids'][0]:
            return json.dumps({"error": "No similar patterns found"})
        
        # Analyze outcomes of similar patterns
        similar_patterns = []
        outcomes = []
        future_returns = []
        
        for i, (id_, metadata, document, distance) in enumerate(zip(
            results['ids'][0], 
            results['metadatas'][0], 
            results['documents'][0],
            results['distances'][0]
        )):
            similarity = max(0, 1 - distance / 10)  # Normalize distance to similarity score
            similar_patterns.append({
                "rank": i + 1,
                "date": metadata['pattern_end_date'],
                "ticker": metadata['ticker'],
                "outcome": metadata['outcome'],
                "future_return_pct": metadata['future_return_pct'],
                "similarity_score": round(similarity, 3),
            })
            outcomes.append(metadata['outcome'])
            future_returns.append(metadata['future_return_pct'])
        
        # Statistical summary of what happened after similar patterns
        avg_future_return = np.mean(future_returns) if future_returns else 0
        positive_outcomes = sum(1 for r in future_returns if r > 0)
        
        from collections import Counter
        outcome_counts = Counter(outcomes)
        
        result = {
            "ticker": ticker.upper(),
            "current_pattern_end": recent_prices.index[-1].strftime("%Y-%m-%d"),
            "similar_patterns_found": len(similar_patterns),
            "historical_outcomes": {
                "avg_future_return_pct": round(avg_future_return, 2),
                "positive_outcome_rate": round(positive_outcomes / len(future_returns) * 100, 1) if future_returns else 0,
                "outcome_distribution": dict(outcome_counts),
            },
            "prediction_signal": "BULLISH" if avg_future_return > 1 else ("BEARISH" if avg_future_return < -1 else "NEUTRAL"),
            "confidence": "HIGH" if len(similar_patterns) >= 5 and abs(avg_future_return) > 2 else "MEDIUM" if len(similar_patterns) >= 3 else "LOW",
            "top_matches": similar_patterns[:5],
        }
        return json.dumps(result, indent=2)
    except Exception as e:
        return f"Error finding similar patterns: {str(e)}"


# Test: Build pattern database and find similar patterns
print("=== Vector Database Pattern Matching ===\n")

# Build the database
print("Building pattern database for AAPL (5 years of history)...")
build_result = build_pattern_database("AAPL", window=20, period="5y")
parsed = json.loads(build_result)
print(f"Patterns stored: {parsed.get('patterns_stored', 'Error')}")

# Find similar patterns to current
print("\nSearching for patterns similar to current AAPL price action...")
similar = find_similar_patterns("AAPL")
parsed_sim = json.loads(similar)
print(f"\nPrediction Signal: {parsed_sim.get('prediction_signal', 'N/A')}")
print(f"Confidence: {parsed_sim.get('confidence', 'N/A')}")
print(f"Avg Future Return: {parsed_sim.get('historical_outcomes', {}).get('avg_future_return_pct', 'N/A')}%")
print(f"Positive Rate: {parsed_sim.get('historical_outcomes', {}).get('positive_outcome_rate', 'N/A')}%")
print(f"\nTop Similar Patterns:")
for match in parsed_sim.get('top_matches', [])[:5]:
    print(f"  {match['date']} | Outcome: {match['outcome']} ({match['future_return_pct']}%) | Similarity: {match['similarity_score']}")

## 16. Guardrails: Safety & Compliance

Guardrails prevent the agent from making dangerous or irresponsible trading suggestions. They validate both **inputs** (what the user asks) and **outputs** (what the agent recommends).

Implemented guardrails:
- **Concentration limit**: No single stock > 25% of portfolio
- **Leverage detection**: Block suggestions involving margin/leverage
- **Penny stock filter**: Warn about stocks under $5
- **Volatility check**: Flag extremely volatile stocks
- **Disclaimer enforcement**: Always include "not financial advice" disclaimer
- **Prohibited actions**: Block insider trading language, pump-and-dump patterns

In [ ]:
class InvestmentGuardrails:
    """Safety guardrails for the stock analysis agent.
    Validates inputs and outputs to prevent dangerous trading suggestions."""
    
    # Prohibited terms that suggest dangerous behavior
    PROHIBITED_INPUT_PATTERNS = [
        "insider", "non-public", "material information",
        "pump and dump", "front run", "manipulate",
        "guaranteed profit", "can't lose", "risk free",
        "all in", "yolo", "bet everything",
    ]
    
    LEVERAGE_KEYWORDS = [
        "margin", "leverage", "borrowed", "options naked",
        "short sell everything", "3x", "leveraged etf",
    ]
    
    MAX_SINGLE_POSITION_PCT = 25  # Max 25% in a single stock
    MIN_STOCK_PRICE = 5.0  # Warn for penny stocks
    MAX_VOLATILITY_ANNUAL = 100  # Flag if annualized vol > 100%
    
    def __init__(self):
        self.violations = []
        self.warnings = []
    
    def reset(self):
        """Clear previous violations and warnings."""
        self.violations = []
        self.warnings = []
    
    def check_input(self, user_query: str) -> dict:
        """Validate user input for prohibited or dangerous requests."""
        self.reset()
        query_lower = user_query.lower()
        
        # Check for prohibited patterns
        for pattern in self.PROHIBITED_INPUT_PATTERNS:
            if pattern in query_lower:
                self.violations.append(
                    f"BLOCKED: Query contains prohibited pattern '{pattern}'. "
                    f"This system cannot assist with potentially illegal or extremely risky activities."
                )
        
        # Check for leverage/margin requests
        for keyword in self.LEVERAGE_KEYWORDS:
            if keyword in query_lower:
                self.warnings.append(
                    f"WARNING: Query mentions '{keyword}'. Leveraged strategies carry "
                    f"significant risk of loss exceeding initial investment."
                )
        
        return {
            "passed": len(self.violations) == 0,
            "violations": self.violations,
            "warnings": self.warnings,
        }
    
    def check_output(self, agent_response: str, context: dict = None) -> dict:
        """Validate agent output for compliance with safety rules."""
        self.reset()
        response_lower = agent_response.lower()
        
        # Check for missing disclaimer
        has_disclaimer = any(phrase in response_lower for phrase in [
            "not financial advice", "educational", "not a recommendation",
            "consult a financial advisor", "do your own research",
            "disclaimer", "for informational purposes"
        ])
        if not has_disclaimer:
            self.warnings.append(
                "GUARDRAIL: Response should include a disclaimer that this is not financial advice."
            )
        
        # Check for absolute language
        absolute_phrases = [
            "guaranteed", "will definitely", "can't fail",
            "sure thing", "risk-free", "no downside",
        ]
        for phrase in absolute_phrases:
            if phrase in response_lower:
                self.warnings.append(
                    f"GUARDRAIL: Response contains overconfident language ('{phrase}'). "
                    f"Investment outcomes are never guaranteed."
                )
        
        # Check concentration recommendations
        if context and "allocation" in context:
            for ticker, allocation in context.get("allocation", {}).items():
                if allocation > self.MAX_SINGLE_POSITION_PCT:
                    self.warnings.append(
                        f"GUARDRAIL: Recommended {allocation}% in {ticker} exceeds "
                        f"max single position limit of {self.MAX_SINGLE_POSITION_PCT}%."
                    )
        
        return {
            "passed": len(self.violations) == 0,
            "violations": self.violations,
            "warnings": self.warnings,
            "disclaimer_present": has_disclaimer,
        }
    
    def check_stock_safety(self, ticker: str) -> dict:
        """Check if a stock meets safety criteria (price, volatility)."""
        self.reset()
        try:
            stock = yf.Ticker(ticker)
            hist = stock.history(period="3mo")
            
            if hist.empty:
                return {"passed": True, "warnings": ["Could not verify stock safety - no data"]}
            
            current_price = hist['Close'].iloc[-1]
            daily_returns = hist['Close'].pct_change().dropna()
            annual_volatility = daily_returns.std() * np.sqrt(252) * 100
            
            # Penny stock check
            if current_price < self.MIN_STOCK_PRICE:
                self.warnings.append(
                    f"PENNY STOCK WARNING: {ticker.upper()} trades at ${current_price:.2f}. "
                    f"Stocks under ${self.MIN_STOCK_PRICE} are highly speculative with low liquidity."
                )
            
            # Extreme volatility check
            if annual_volatility > self.MAX_VOLATILITY_ANNUAL:
                self.warnings.append(
                    f"HIGH VOLATILITY WARNING: {ticker.upper()} has {annual_volatility:.0f}% "
                    f"annualized volatility. This is extremely risky."
                )
            
            # Volume check
            avg_volume = hist['Volume'].mean()
            if avg_volume < 100000:
                self.warnings.append(
                    f"LOW LIQUIDITY WARNING: {ticker.upper()} averages only {avg_volume:,.0f} "
                    f"shares/day. May be difficult to exit positions."
                )
            
            return {
                "passed": len(self.violations) == 0,
                "ticker": ticker.upper(),
                "current_price": round(current_price, 2),
                "annual_volatility_pct": round(annual_volatility, 1),
                "avg_daily_volume": int(avg_volume),
                "warnings": self.warnings,
            }
        except Exception as e:
            return {"passed": True, "warnings": [f"Safety check error: {str(e)}"]}
    
    def add_disclaimer(self, response: str) -> str:
        """Append a standard disclaimer if one is not already present."""
        if not any(phrase in response.lower() for phrase in ["not financial advice", "disclaimer", "educational"]):
            response += "\n\n---\n*DISCLAIMER: This analysis is for educational and informational purposes only. It is not financial advice. Always do your own research and consult a qualified financial advisor before making investment decisions.*"
        return response


# Initialize guardrails
guardrails = InvestmentGuardrails()

# Test guardrails
print("=== Guardrails Testing ===\n")

# Test input validation
print("1. Testing input validation:")
tests = [
    "Analyze AAPL for me",  # Should pass
    "I have insider information about MSFT earnings",  # Should block
    "Should I use margin to buy NVDA?",  # Should warn
    "Put all my money in one stock, guaranteed profit",  # Should block
]

for test in tests:
    result = guardrails.check_input(test)
    status = "PASS" if result['passed'] else "BLOCKED"
    print(f"  [{status}] \"{test[:50]}...\"")
    for v in result['violations']:
        print(f"       {v}")
    for w in result['warnings']:
        print(f"       {w}")

# Test output validation
print("\n2. Testing output validation:")
good_response = "Based on the analysis, AAPL looks fairly valued. This is not financial advice - do your own research."
bad_response = "This stock is a guaranteed winner. You can't lose money on this. Put 50% of your portfolio in it."

for resp in [good_response, bad_response]:
    result = guardrails.check_output(resp)
    print(f"  Response: \"{resp[:60]}...\"")
    print(f"  Disclaimer present: {result['disclaimer_present']}")
    for w in result['warnings']:
        print(f"       {w}")
    print()

# Test stock safety
print("3. Testing stock safety check:")
safety = guardrails.check_stock_safety("AAPL")
print(f"  AAPL: Price=${safety.get('current_price', 'N/A')}, Vol={safety.get('annual_volatility_pct', 'N/A')}%")
print(f"  Warnings: {safety['warnings'] if safety['warnings'] else 'None'}")

## 17. Complete Agent: All Tools Integrated with Guardrails

This section brings together ALL tools (original + new) into a single comprehensive agent that:
- Has access to 12+ tools spanning price data, fundamentals, technicals, sentiment, filings, backtesting, optimization, and pattern matching
- Applies guardrails on every input and output
- Provides well-rounded investment analysis with safety checks

In [ ]:
# Complete tool registry with all tools
ALL_TOOLS = {
    # Original tools
    "get_stock_price": get_stock_price,
    "get_fundamentals": get_fundamentals,
    "get_technical_indicators": get_technical_indicators,
    "get_analyst_recommendations": get_analyst_recommendations,
    "get_historical_performance": get_historical_performance,
    "compare_stocks": compare_stocks,
    # New tools
    "get_news_sentiment": get_news_sentiment,
    "get_llm_news_analysis": get_llm_news_analysis,
    "get_sec_filings": get_sec_filings,
    "get_filing_summary": get_filing_summary,
    "optimize_portfolio": optimize_portfolio,
    "backtest_strategy": backtest_strategy,
    "build_pattern_database": build_pattern_database,
    "find_similar_patterns": find_similar_patterns,
}

COMPLETE_AGENT_SYSTEM_PROMPT = """You are an expert stock investment analyst AI agent with comprehensive tools for analysis.

AVAILABLE TOOLS:
--- Price & Market Data ---
- get_stock_price(ticker): Current price and daily change
- get_fundamentals(ticker): P/E, market cap, margins, growth, debt ratios
- get_technical_indicators(ticker): SMA, RSI, MACD, Bollinger Bands
- get_analyst_recommendations(ticker): Analyst consensus and price targets
- get_historical_performance(ticker): Returns, volatility, Sharpe, max drawdown
- compare_stocks(tickers): Compare multiple stocks (comma-separated, e.g., "AAPL,MSFT,GOOGL")

--- Sentiment & Filings ---
- get_news_sentiment(ticker): Recent news headlines with NLP sentiment scores
- get_llm_news_analysis(ticker): Deep LLM-based news interpretation
- get_sec_filings(ticker): List recent SEC 10-K/10-Q filings with links
- get_filing_summary(ticker): AI-generated summary of latest 10-K filing

--- Portfolio & Strategy ---
- optimize_portfolio(tickers): Mean-variance optimization (comma-separated tickers)
- backtest_strategy(ticker,strategy): Backtest strategies: sma_crossover, rsi_mean_reversion, buy_and_hold

--- Pattern Analysis ---
- build_pattern_database(ticker): Build vector DB of historical price patterns (run before find_similar_patterns)
- find_similar_patterns(ticker): Find current patterns similar to historical ones and predict outcomes

RESEARCH WORKFLOW:
1. Gather price and fundamental data
2. Check technical momentum
3. Analyze news sentiment
4. Review SEC filings for material information
5. Check historical patterns for prediction signals
6. If portfolio question: run optimization
7. If strategy question: run backtesting
8. Synthesize into a comprehensive recommendation

RESPONSE FORMAT:
THOUGHT: [your reasoning about what to do next]
ACTION: [tool_name]
ACTION_INPUT: [input to the tool]

When ready with final analysis:
THOUGHT: [synthesis of all data]
FINAL_ANSWER: [comprehensive analysis with:
- Key Findings
- Valuation Assessment
- Sentiment & News Summary
- Technical Outlook
- Risk Factors
- Historical Pattern Signal (if available)
- Recommendation: STRONG BUY / BUY / HOLD / SELL / STRONG SELL
- Confidence Level: HIGH / MEDIUM / LOW

DISCLAIMER: This is educational AI analysis, not financial advice.]

IMPORTANT RULES:
- Always use tools for REAL data. Never fabricate numbers.
- Consider multiple perspectives before recommending.
- Acknowledge risks and uncertainties.
- Always include a disclaimer in your final answer.
- If data is unavailable, state that clearly."""


def run_complete_agent(query: str, max_iterations: int = 12, verbose: bool = True) -> str:
    """Run the complete investment agent with all tools and guardrails."""
    
    # INPUT GUARDRAIL CHECK
    input_check = guardrails.check_input(query)
    
    if not input_check['passed']:
        blocked_msg = "BLOCKED BY GUARDRAILS:\n" + "\n".join(input_check['violations'])
        if verbose:
            print(blocked_msg)
        return blocked_msg
    
    if input_check['warnings'] and verbose:
        print("⚠️  GUARDRAIL WARNINGS:")
        for w in input_check['warnings']:
            print(f"  {w}")
        print()
    
    messages = [
        {"role": "system", "content": COMPLETE_AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": query}
    ]
    
    if verbose:
        print(f"{'='*70}")
        print(f"COMPREHENSIVE STOCK ANALYSIS AGENT")
        print(f"{'='*70}")
        print(f"Query: {query}\n")
    
    for i in range(max_iterations):
        response = ollama.chat(model=MODEL, messages=messages)
        assistant_text = response['message']['content']
        thought, action, action_input, final_answer = parse_agent_response(assistant_text)
        
        if verbose:
            print(f"--- Step {i+1} ---")
            if thought:
                print(f"  Thought: {thought[:100]}")
        
        if final_answer:
            # OUTPUT GUARDRAIL CHECK
            output_check = guardrails.check_output(final_answer)
            
            if output_check['warnings'] and verbose:
                print("\n⚠️  OUTPUT GUARDRAIL WARNINGS:")
                for w in output_check['warnings']:
                    print(f"  {w}")
            
            # Add disclaimer if missing
            final_answer = guardrails.add_disclaimer(final_answer)
            
            if verbose:
                print(f"\n{'='*70}")
                print("ANALYSIS COMPLETE")
                print(f"{'='*70}")
                print(final_answer)
            
            return final_answer
        
        if action and action in ALL_TOOLS:
            if verbose:
                print(f"  Action: {action}({action_input})")
            
            # Stock safety check for relevant tools
            if action in ["get_stock_price", "get_fundamentals", "get_technical_indicators"]:
                ticker_to_check = action_input.split(",")[0].strip()
                safety = guardrails.check_stock_safety(ticker_to_check)
                if safety['warnings'] and verbose:
                    for w in safety['warnings']:
                        print(f"  ⚠️ {w}")
            
            observation = ALL_TOOLS[action](action_input)
            
            if verbose:
                obs_preview = observation[:150] + "..." if len(observation) > 150 else observation
                print(f"  Result: {obs_preview}\n")
            
            messages.append({"role": "assistant", "content": assistant_text})
            messages.append({"role": "user", "content": f"OBSERVATION: {observation}\n\nContinue analysis or provide FINAL_ANSWER."})
        elif action:
            if verbose:
                print(f"  Unknown tool: {action}")
            messages.append({"role": "assistant", "content": assistant_text})
            messages.append({"role": "user", "content": f"Error: Unknown tool '{action}'. Available: {list(ALL_TOOLS.keys())}"})
        else:
            # No structured response - use as final answer
            final_answer = guardrails.add_disclaimer(assistant_text)
            if verbose:
                print(f"\n{final_answer}")
            return final_answer
    
    return "Agent reached maximum iterations. Try a more specific question."


print("Complete Agent initialized with ALL tools and guardrails!")
print(f"\nAvailable tools ({len(ALL_TOOLS)}):")
for name in ALL_TOOLS:
    print(f"  • {name}")

In [ ]:
# Example 1: Full analysis of a single stock (uses sentiment, technicals, fundamentals, patterns)
result = run_complete_agent(
    "Give me a comprehensive analysis of NVDA including news sentiment, "
    "technical indicators, fundamental valuation, and check if current "
    "price patterns match any historical patterns. What's your recommendation?"
)

In [ ]:
# Example 2: Portfolio optimization with backtesting
result = run_complete_agent(
    "I want to build a portfolio with AAPL, MSFT, GOOGL, AMZN, and JPM. "
    "Optimize the allocation using Modern Portfolio Theory, then backtest "
    "the SMA crossover strategy on the top pick. What allocation do you recommend?"
)

In [ ]:
# Example 3: Guardrails in action - this should be blocked
result = run_complete_agent(
    "I have insider information about a company's earnings. "
    "Tell me how to trade on it for guaranteed profit."
)

In [ ]:
# Example 4: SEC filings analysis
result = run_complete_agent(
    "Review MSFT's recent SEC filings and summarize the key points from their latest 10-K. "
    "What does the filing tell us about their financial health and growth outlook?"
)

## 18. Key Concepts Recap

### Complete Tool Inventory

| Category | Tool | Purpose |
|----------|------|---------|
| **Market Data** | `get_stock_price` | Current price & daily change |
| **Market Data** | `get_historical_performance` | Returns, volatility, Sharpe, drawdown |
| **Fundamentals** | `get_fundamentals` | P/E, margins, growth, debt |
| **Fundamentals** | `get_sec_filings` | SEC EDGAR 10-K/10-Q retrieval |
| **Fundamentals** | `get_filing_summary` | LLM-generated 10-K summary |
| **Technicals** | `get_technical_indicators` | SMA, RSI, MACD, Bollinger |
| **Sentiment** | `get_news_sentiment` | NLP sentiment on headlines |
| **Sentiment** | `get_llm_news_analysis` | Deep LLM news interpretation |
| **Strategy** | `backtest_strategy` | Validate strategies on history |
| **Strategy** | `optimize_portfolio` | Mean-variance optimization (MPT) |
| **Patterns** | `build_pattern_database` | Store patterns in vector DB |
| **Patterns** | `find_similar_patterns` | Find similar historical patterns |
| **Safety** | `InvestmentGuardrails` | Input/output validation |

### Architecture Patterns Used

1. **ReAct Agent** (Sections 4-6): Think → Act → Observe → Repeat
2. **Multi-Agent System** (Section 7): Specialist analysts → Portfolio Manager coordinator
3. **Conversational Agent** (Section 8): Stateful memory across questions
4. **NLP Pipeline** (Section 11): TextBlob + LLM for dual-layer sentiment
5. **External API Integration** (Section 12): SEC EDGAR for regulatory filings
6. **Mathematical Optimization** (Section 13): Scipy for portfolio optimization
7. **Strategy Backtesting** (Section 14): Historical validation of trading rules
8. **Vector Similarity Search** (Section 15): ChromaDB for pattern matching
9. **Guardrails/Safety Layer** (Section 16): Input/output validation & compliance

### What Makes This "Agentic"?

| Feature | Traditional LLM | This Agentic System |
|---------|----------------|---------------------|
| Data Access | Only training data (stale) | 14 tools fetching live data |
| Reasoning | Single-pass response | Multi-step reasoning loop |
| Planning | None | Autonomous research workflow |
| Safety | None | Guardrails on inputs & outputs |
| Validation | No backtesting | Historical strategy validation |
| Pattern Recognition | None | Vector DB similarity search |
| Optimization | None | Mathematical portfolio optimization |
| Multi-perspective | Single viewpoint | Multiple specialist agents |
| Memory | Stateless | Context across conversations |

### Running with Different Models

```python
# Try different models for different trade-offs:
MODEL = "llama3.1"      # Good balance of capability and speed
MODEL = "mistral"       # Fast, good for simple analysis
MODEL = "llama3.1:70b"  # More capable but slower (needs more RAM)
MODEL = "deepseek-r1"   # Strong reasoning, good for complex analysis
MODEL = "qwen2.5"       # Good multilingual support
```

### Important Disclaimer

This notebook is for **educational purposes only**. AI-generated stock analysis should never be the sole basis for investment decisions. Always:
- Do your own research
- Consult qualified financial advisors
- Understand that past performance doesn't guarantee future results
- Be aware that LLMs can hallucinate or misinterpret data
- Never trade on non-public information